## TinyEHR Diabates Prediction with Randon Forest

#### 1. Import packages

In [1]:
import sqlite3 as sql3
import pandas as pd
import os
import tinyehr

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#### 2. Connect to TinyEHR

In [2]:
tinyehr_db = r'C:\tinyehr\tinyehr_mimic_format.db'

if not os.path.exists(tinyehr_db):
    tinyehr.build_sqlite(format='tinyehr_mimic_format', output='tinyehr_db')

con = sql3.connect(tinyehr_db)

#### 3. Explore related tables

In [3]:
query = 'select name from sqlite_master where type="table" order by name'
tables = pd.read_sql(query, con)
tables

,name
0,admissions
1,caregiver
2,chartevents
3,d_hcpcs
4,d_icd_diagnoses
5,d_icd_procedures
6,d_items
7,d_labitems
8,date_offsets
9,datetimeevents


#### 4. Load patients

In [4]:
query = 'select * from patients'
patients = pd.read_sql(query, con)
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2013,2011 - 2013,None
1,10003400,F,72,2011,2011 - 2013,2014-09-02 00:00:00
2,10002428,F,80,2011,2011 - 2013,None
3,10032725,F,38,2013,2011 - 2013,2013-03-30 00:00:00
4,10027445,F,48,2012,2011 - 2013,2016-02-09 00:00:00


Let's inspect some information on patients:

In [5]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   subject_id         100 non-null    int64 
 1   gender             100 non-null    object
 2   anchor_age         100 non-null    int64 
 3   anchor_year        100 non-null    int64 
 4   anchor_year_group  100 non-null    object
 5   dod                31 non-null     object
dtypes: int64(3), object(3)
memory usage: 4.8+ KB


Check which columns exist:

In [6]:
patients.columns.tolist()

['subject_id',
 'gender',
 'anchor_age',
 'anchor_year',
 'anchor_year_group',
 'dod']

#### 5. Find diagnoses of diabetes

In [7]:
query = 'select distinct icd_code, long_title from d_icd_diagnoses where lower(long_title) like "%diabete%" order by icd_code'
diabetes_codes = pd.read_sql(query, con)
diabetes_codes

,icd_code,long_title
0,249.00,Secondary diabetes mellitus without mention of...
1,249.01,Secondary diabetes mellitus without mention of...
2,249.10,"Secondary diabetes mellitus with ketoacidosis,..."
3,249.11,"Secondary diabetes mellitus with ketoacidosis,..."
4,249.20,Secondary diabetes mellitus with hyperosmolari...
...,...,...
709,V18.0,Family history of diabetes mellitus
710,V77.1,Screening for diabetes mellitus
711,Z13.1,Encounter for screening for diabetes mellitus
712,Z83.3,Family history of diabetes mellitus


#### 6. Create a target variable for diabetes

Get the patient ids:

In [8]:
query = """
select distinct di.subject_id 
from diagnoses_icd di 
join d_icd_diagnoses dd 
on di.icd_code = dd.icd_code 
where lower(dd.long_title) like '%diabete%'
"""

diabetes_patients = pd.read_sql(query, con)
diabetes_patients

,subject_id
0,10009628
1,10015860
2,10029484
3,10037928
4,10007795
5,10005817
6,10006580
7,10012853
8,10014078
9,10016150


Create the target - "diabetes":

In [9]:
target = 'diabetes'
patients[target] = patients['subject_id'].isin(diabetes_patients['subject_id']).astype(int)
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,diabetes
0,10014729,F,21,2013,2011 - 2013,None,0
1,10003400,F,72,2011,2011 - 2013,2014-09-02 00:00:00,0
2,10002428,F,80,2011,2011 - 2013,None,0
3,10032725,F,38,2013,2011 - 2013,2013-03-30 00:00:00,1
4,10027445,F,48,2012,2011 - 2013,2016-02-09 00:00:00,1


Count diabates v.s. non-diabetes:

In [10]:
patients['diabetes'].value_counts()

diabetes
0    65
1    35
Name: count, dtype: int64

#### 7. Find glucose measurements

Search glucose in the dictionary table:

In [11]:
query = 'select * from d_labitems where lower(label) like "%glucose%"'
glucose_items = pd.read_sql(query, con)
glucose_items

,itemid,label,fluid,category
0,52027,"Glucose, Whole Blood",Blood,Blood Gas
1,50809,Glucose,Blood,Blood Gas
2,50931,Glucose,Blood,Chemistry
3,52569,Glucose,Blood,Chemistry
4,51941,"Glucose, Stool",Stool,Chemistry
5,51084,"Glucose, Urine",Urine,Chemistry
6,51981,Glucose,Urine,Chemistry
7,50842,"Glucose, Ascites",Ascites,Chemistry
8,51053,"Glucose, Pleural",Pleural,Chemistry
9,51022,"Glucose, Joint Fluid",Joint Fluid,Chemistry


#### 8. Extract glucose results

Pick one itemid from the result, and search it from the LABEVENTS table:

In [12]:
glucose_itemid = 50931
query = f'select subject_id, valuenum from labevents where itemid = {glucose_itemid}'
glucose = pd.read_sql(query, con)
glucose

,subject_id,valuenum
0,10014354,193.0
1,10014354,210.0
2,10035631,168.0
3,10029484,415.0
4,10035631,97.0
...,...,...
2706,10035631,129.0
2707,10035631,158.0
2708,10021487,116.0
2709,10040025,157.0


#### 9. Clean glucose data

Check missing values:

In [13]:
glucose.isnull().sum()

subject_id    0
valuenum      0
dtype: int64

No missing values

In [14]:
glucose.describe()

,subject_id,valuenum
count,2.711000e+03,2711.000000
mean,1.002096e+07,144.029878
std,1.198478e+04,75.964433
min,1.000003e+07,29.000000
25%,1.001422e+07,99.000000
50%,1.001978e+07,121.000000
75%,1.003272e+07,164.000000
max,1.004002e+07,996.000000


All values of valuenum are between 29 and 996.

#### 10. Aggregate by patient

Calculate glucose stats including mean, max and min for each patient:

In [15]:
glucose_features = (
    glucose.groupby('subject_id')['valuenum']
           .agg(
               glucose_mean='mean',
               glucose_max='max',
               glucose_min='min'
           )
           .reset_index()
)

glucose_features.head()

,subject_id,glucose_mean,glucose_max,glucose_min
0,10000032,106.176471,155.0,71.0
1,10001217,96.363636,113.0,86.0
2,10001725,126.285714,152.0,92.0
3,10002428,108.400000,183.0,67.0
4,10002495,252.900000,370.0,165.0


#### 11. Build modeling dataset

Let's merge patients and labs:

In [16]:
model_df = patients.merge(glucose_features, on='subject_id', how='left')
model_df.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,diabetes,glucose_mean,glucose_max,glucose_min
0,10014729,F,21,2013,2011 - 2013,None,0,95.315789,140.0,74.0
1,10003400,F,72,2011,2011 - 2013,2014-09-02 00:00:00,0,113.030612,159.0,70.0
2,10002428,F,80,2011,2011 - 2013,None,0,108.400000,183.0,67.0
3,10032725,F,38,2013,2011 - 2013,2013-03-30 00:00:00,1,173.689655,266.0,80.0
4,10027445,F,48,2012,2011 - 2013,2016-02-09 00:00:00,1,209.217391,525.0,50.0


#### 12. Check missing values

In [17]:
model_df['glucose_mean'].isnull().sum()

0

In [18]:
model_df['glucose_max'].isnull().sum()

0

In [19]:
model_df['glucose_min'].isnull().sum()

0

No missing values from glucose mean, max and min.

#### 13. Encode gender

In [20]:
model_df['gender']

0     F
1     F
2     F
3     F
4     F
     ..
95    M
96    M
97    M
98    M
99    M
Name: gender, Length: 100, dtype: object

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
model_df['gender'] = le.fit_transform(model_df['gender'])
model_df

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,diabetes,glucose_mean,glucose_max,glucose_min
0,10014729,0,21,2013,2011 - 2013,None,0,95.315789,140.0,74.0
1,10003400,0,72,2011,2011 - 2013,2014-09-02 00:00:00,0,113.030612,159.0,70.0
2,10002428,0,80,2011,2011 - 2013,None,0,108.400000,183.0,67.0
3,10032725,0,38,2013,2011 - 2013,2013-03-30 00:00:00,1,173.689655,266.0,80.0
4,10027445,0,48,2012,2011 - 2013,2016-02-09 00:00:00,1,209.217391,525.0,50.0
...,...,...,...,...,...,...,...,...,...,...
95,10004733,1,51,2014,2014 - 2016,None,0,94.913043,125.0,73.0
96,10021118,1,62,2014,2014 - 2016,None,0,106.250000,117.0,98.0
97,10018501,1,83,2015,2014 - 2016,None,0,126.000000,158.0,99.0
98,10007058,1,48,2015,2014 - 2016,None,0,102.500000,111.0,92.0


#### 14. Select features

We want to analyze if the following features, like age, gender and glucose-related features, can predict diabetes.

In [22]:
features = ['anchor_age', 'gender', 'glucose_mean', 'glucose_max', 'glucose_min']
X = model_df[features]
y = model_df[target]

In [23]:
X

,anchor_age,gender,glucose_mean,glucose_max,glucose_min
0,21,0,95.315789,140.0,74.0
1,72,0,113.030612,159.0,70.0
2,80,0,108.400000,183.0,67.0
3,38,0,173.689655,266.0,80.0
4,48,0,209.217391,525.0,50.0
...,...,...,...,...,...
95,51,1,94.913043,125.0,73.0
96,62,1,106.250000,117.0,98.0
97,83,1,126.000000,158.0,99.0
98,48,1,102.500000,111.0,92.0


In [24]:
y

0     0
1     0
2     0
3     1
4     1
     ..
95    0
96    0
97    0
98    0
99    1
Name: diabetes, Length: 100, dtype: int32

#### 15. Split train / test datasets

In [25]:
rs = 22

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs, stratify=y)

#### 16. Train model and predict

We set 100 decision trees in the forest.

In [27]:
RF = RandomForestClassifier(n_estimators=100, random_state=rs)

In [28]:
RF.fit(X_train, y_train)

RandomForestClassifier(random_state=22)

In [29]:
y_pred = RF.predict(X_test)

#### 17. Evaluate model

In [30]:
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

Accuracy: 0.75


The accuracy tells that 75% of patients are classified correctly.

In [31]:
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.79      0.85      0.81        13
           1       0.67      0.57      0.62         7

    accuracy                           0.75        20
   macro avg       0.73      0.71      0.72        20
weighted avg       0.74      0.75      0.75        20



Let's take a look at the report:
- The lowest value is the recall. The model only found 57% of all diabetic patients.
- The precision says that the model's prediction is correct 67% of the time.
- The F1 score means the model's ability to detect diabetics is just above the average.
- Based on the support values, 13 / (13 + 7) = 0.65, 65% of non-diabetic; and 7 / (13 + 7) = 0.35, 35% of diabetic, a slightly imbalanced.

In [32]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[11  2]
 [ 3  4]]


Notice that there are 3 missed diabetics.

#### 18. Feature importance

In [33]:
importance = pd.DataFrame({'feature': X.columns, 'importance': RF.feature_importances_})
importance

,feature,importance
0,anchor_age,0.100472
1,gender,0.035268
2,glucose_mean,0.416721
3,glucose_max,0.254855
4,glucose_min,0.192684


In [34]:
importance.sort_values('importance', ascending=False)

,feature,importance
2,glucose_mean,0.416721
3,glucose_max,0.254855
4,glucose_min,0.192684
0,anchor_age,0.100472
1,gender,0.035268


Among the 5 features that we selected, the sum of mean, max and min of glucose is about 86% (41.6% + 25.4% + 19.2%) while age about 10% and gender about 3.5%. The model's prediction mostly relies on the glucose-related features. The age and gender does not contribute much. For the future research we need to involve more features and make the dataset a little more balanced.

#### 19. Save final dataset

In [35]:
model_df.to_csv(r'C:\tinyehr\tinyehr_diabetes_dataset.csv', index=False)